# 🇧🇹 Bhutanese Language Speech Model Training
## Whisper Large-v3 Fine-tuning for Dzongkha (རྫོང་ཁ) Language Extraction

**Target Hardware:** 32GB VRAM (e.g., NVIDIA A100 / RTX 4090 x2 / RTX 6000 Ada)

**Model:** `openai/whisper-large-v3` — Best ASR model that fits and maximally utilizes 32GB VRAM

**Task:** Automatic Speech Recognition (ASR) + Language Extraction from Bhutanese (Dzongkha) documents/audio

---
### Why Whisper Large-v3?
- **1.5B parameters** — fits comfortably in 32GB VRAM (~6–8GB model weights, leaving room for batching)
- Best multilingual ASR model available with native Dzongkha support
- Superior performance on low-resource languages like Dzongkha
- Supports 99 languages including Tibetan script variants used in Bhutan

### VRAM Budget (32GB)
| Component | Estimated VRAM |
|-----------|---------------|
| Model weights (bf16) | ~6 GB |
| Optimizer states (AdamW) | ~12 GB |
| Activations + gradients | ~8 GB |
| Batch overhead | ~4 GB |
| **Total** | **~30 GB** |


## 1. Install Dependencies

In [ ]:
%%bash
pip install -q \
    transformers>=4.40.0 \
    datasets>=2.18.0 \
    accelerate>=0.28.0 \
    peft>=0.10.0 \
    bitsandbytes>=0.43.0 \
    evaluate>=0.4.0 \
    jiwer>=3.0.0 \
    librosa>=0.10.0 \
    soundfile>=0.12.0 \
    tqdm \
    tensorboard \
    PyMuPDF \
    pytesseract \
    Pillow \
    torch>=2.2.0 \
    torchaudio>=2.2.0

echo "✅ All packages installed"

## 2. Imports & Environment Setup

In [ ]:
import os
import re
import json
import warnings
import unicodedata
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

import torch
import numpy as np
import librosa
import soundfile as sf
from tqdm.auto import tqdm

from datasets import (
    load_dataset, Dataset, DatasetDict,
    Audio, concatenate_datasets
)
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    WhisperFeatureExtractor,
    WhisperTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel
)
import evaluate

warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device check ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device : {device}")

if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"💾  VRAM   : {total_vram:.1f} GB")
    assert total_vram >= 20, "⚠️  Less than 20 GB VRAM detected — reduce batch size or enable 8-bit"

print("✅ Environment ready")

## 3. Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                     GLOBAL CONFIGURATION                        ║
# ╚══════════════════════════════════════════════════════════════════╝

CFG = {
    # ── Model ─────────────────────────────────────────────────────
    "model_name"        : "openai/whisper-large-v3",   # Best model for 32GB VRAM
    "language"          : "dzongkha",                  # Bhutan's official language
    "task"              : "transcribe",

    # ── LoRA (Parameter-Efficient Fine-Tuning) ─────────────────────
    # Reduces trainable params from 1.5B → ~20M while keeping accuracy
    "use_lora"          : True,
    "lora_r"            : 32,
    "lora_alpha"        : 64,
    "lora_dropout"      : 0.05,

    # ── Audio ──────────────────────────────────────────────────────
    "sample_rate"       : 16_000,      # Whisper expects 16kHz
    "max_audio_len_s"   : 30,          # Max clip length in seconds

    # ── Training ──────────────────────────────────────────────────
    "output_dir"        : "./whisper-dzongkha-bhutan",
    "per_device_train_batch_size" : 8,    # Tuned for 32GB VRAM
    "per_device_eval_batch_size"  : 8,
    "gradient_accumulation_steps": 2,    # Effective batch = 16
    "learning_rate"     : 1e-4,
    "warmup_steps"      : 500,
    "max_steps"         : 8_000,
    "eval_steps"        : 500,
    "save_steps"        : 500,
    "logging_steps"     : 50,
    "fp16"              : False,
    "bf16"              : True,          # A100 / Ada support bfloat16
    "gradient_checkpointing": True,      # Saves ~30% VRAM
    "dataloader_num_workers": 4,

    # ── Data ──────────────────────────────────────────────────────
    "data_dir"          : "./data/bhutan",
    "train_split"       : 0.85,
    "val_split"         : 0.10,
    "test_split"        : 0.05,

    # ── Document OCR ──────────────────────────────────────────────
    "docs_dir"          : "./data/bhutan_docs",  # PDF / image documents
    "ocr_lang"          : "dzo+eng",             # Tesseract: Dzongkha + English
}

Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)
Path(CFG["data_dir"]).mkdir(parents=True, exist_ok=True)
Path(CFG["docs_dir"]).mkdir(parents=True, exist_ok=True)

print("📋 Configuration loaded")
print(json.dumps({k: v for k, v in CFG.items() if k != "output_dir"}, indent=2))

## 4. Document Text Extraction (Bhutanese PDFs / Images)

In [ ]:
import fitz          # PyMuPDF
import pytesseract
from PIL import Image
import io


# ══════════════════════════════════════════════════════════════════════════
#  DOCUMENT LANGUAGE EXTRACTION
# ══════════════════════════════════════════════════════════════════════════

def extract_text_from_pdf(pdf_path: str, ocr_lang: str = "dzo+eng") -> str:
    """
    Extract Dzongkha / English text from a Bhutanese PDF document.
    Strategy:
      1. Try native PDF text layer (fast)
      2. Fall back to page-rasterisation + Tesseract OCR (for scanned docs)
    """
    doc = fitz.open(pdf_path)
    all_text = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text").strip()

        if len(text) < 20:  # Likely scanned image — use OCR
            pix = page.get_pixmap(dpi=300)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            text = pytesseract.image_to_string(
                img,
                lang=ocr_lang,
                config="--oem 1 --psm 3"  # LSTM engine, auto-page-segmentation
            ).strip()

        if text:
            all_text.append(text)

    doc.close()
    return "\n".join(all_text)


def extract_text_from_image(img_path: str, ocr_lang: str = "dzo+eng") -> str:
    """OCR a single image file (JPEG, PNG, TIFF)."""
    img = Image.open(img_path)
    return pytesseract.image_to_string(
        img, lang=ocr_lang, config="--oem 1 --psm 3"
    ).strip()


def batch_extract_documents(docs_dir: str, ocr_lang: str = "dzo+eng") -> List[Dict]:
    """
    Walk `docs_dir` and extract text from all .pdf / image files.
    Returns list of {filename, text} dicts.
    """
    docs_path = Path(docs_dir)
    supported_ext = {".pdf", ".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp"}
    records = []

    files = [f for f in docs_path.rglob("*") if f.suffix.lower() in supported_ext]
    print(f"📂 Found {len(files)} document(s) in {docs_dir}")

    for fp in tqdm(files, desc="Extracting documents"):
        try:
            if fp.suffix.lower() == ".pdf":
                text = extract_text_from_pdf(str(fp), ocr_lang)
            else:
                text = extract_text_from_image(str(fp), ocr_lang)

            if text:
                records.append({"filename": fp.name, "text": text})
        except Exception as e:
            print(f"  ⚠️  Skipping {fp.name}: {e}")

    print(f"✅ Extracted text from {len(records)} document(s)")
    return records


# ── Run extraction ────────────────────────────────────────────────────────
extracted_docs = batch_extract_documents(CFG["docs_dir"], CFG["ocr_lang"])

# Save extracted corpus
corpus_path = Path(CFG["data_dir"]) / "bhutan_corpus.json"
with open(corpus_path, "w", encoding="utf-8") as f:
    json.dump(extracted_docs, f, ensure_ascii=False, indent=2)

print(f"💾 Corpus saved → {corpus_path}")

## 5. Data Cleaning
> **Note:** The advanced cleaning block below is intentionally **commented out**. 
> Activate it when you encounter noisy / web-scraped transcripts.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  BASIC CLEANING  (always active)
# ══════════════════════════════════════════════════════════════════════════

# Dzongkha Unicode block: U+0F00–U+0FFF (Tibetan script)
DZONGKHA_RANGE = re.compile(r"[\u0F00-\u0FFF]+")
LATIN_RANGE    = re.compile(r"[A-Za-z]+")

def basic_clean(text: str) -> str:
    """
    Light cleaning suitable for well-formed Dzongkha transcripts.
    - Normalise unicode (NFC)
    - Collapse whitespace
    - Remove null bytes
    """
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\x00", "")      # null bytes
    text = re.sub(r"\s+", " ", text)     # collapse whitespace
    text = text.strip()
    return text


# ══════════════════════════════════════════════════════════════════════════
#  ADVANCED CLEANING  ←── DEACTIVATED  (comment-in when needed)
# ══════════════════════════════════════════════════════════════════════════

# def advanced_clean(text: str, keep_latin: bool = True) -> str:
#     """
#     Aggressive cleaning for noisy / web-scraped Bhutanese transcripts.
#
#     Steps:
#       1. Basic unicode normalisation
#       2. Strip HTML / XML tags
#       3. Remove URLs and email addresses
#       4. Remove numeric-only tokens (page numbers, footnote refs)
#       5. Strip parenthetical asides  (e.g. "[inaudible]", "(background noise)")
#       6. Optionally remove Latin/English tokens (pure Dzongkha mode)
#       7. Deduplicate consecutive identical lines
#       8. Drop lines shorter than 3 characters
#
#     Parameters
#     ----------
#     text        : raw transcript / OCR output string
#     keep_latin  : if False, strips all ASCII Latin characters
#                   (use for monolingual Dzongkha-only training)
#     """
#     # 1. Basic normalise
#     text = unicodedata.normalize("NFC", text)
#     text = text.replace("\x00", "")
#
#     # 2. Strip HTML tags
#     text = re.sub(r"<[^>]+>", " ", text)
#
#     # 3. Remove URLs & emails
#     text = re.sub(r"https?://\S+", "", text)
#     text = re.sub(r"\S+@\S+\.\S+", "", text)
#
#     # 4. Remove standalone numbers (page refs, footnotes)
#     text = re.sub(r"\b\d+\b", "", text)
#
#     # 5. Strip bracketed / parenthetical noise
#     text = re.sub(r"\[.*?\]", "", text)   # [inaudible], [music]
#     text = re.sub(r"\(.*?\)", "", text)   # (background noise)
#
#     # 6. Optionally remove Latin tokens
#     if not keep_latin:
#         text = re.sub(r"[A-Za-z]+", "", text)
#
#     # 7 & 8. Line-level dedup + short-line removal
#     lines = text.splitlines()
#     seen, clean_lines = set(), []
#     for line in lines:
#         line = re.sub(r"\s+", " ", line).strip()
#         if len(line) >= 3 and line not in seen:
#             clean_lines.append(line)
#             seen.add(line)
#
#     return "\n".join(clean_lines)
#
#
# def clean_dataset_advanced(examples):
#     """
#     HuggingFace map-compatible wrapper for advanced_clean.
#     Usage:
#         dataset = dataset.map(clean_dataset_advanced, batched=False)
#     """
#     examples["sentence"] = advanced_clean(
#         examples["sentence"],
#         keep_latin=True   # ← set False for pure Dzongkha mode
#     )
#     return examples


# ── Apply basic cleaning to extracted corpus ──────────────────────────────
for rec in extracted_docs:
    rec["text"] = basic_clean(rec["text"])

print("✅ Basic cleaning applied to all documents")
if extracted_docs:
    sample = extracted_docs[0]
    print(f"\n📄 Sample ({sample['filename']}) — first 300 chars:")
    print(sample["text"][:300])

## 6. Load & Prepare Audio Dataset

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  OPTION A — Load from HuggingFace Hub (if Dzongkha dataset available)
# ══════════════════════════════════════════════════════════════════════════
#
# Uncomment if using a public dataset (e.g., Google FLEURS dzo_BT):
#
# raw_datasets = load_dataset(
#     "google/fleurs",
#     "dzo_BT",              # Dzongkha / Bhutan locale
#     trust_remote_code=True
# )
# print(raw_datasets)


# ══════════════════════════════════════════════════════════════════════════
#  OPTION B — Load from local audio files  (default path)
# ══════════════════════════════════════════════════════════════════════════

def load_local_audio_dataset(data_dir: str, sample_rate: int = 16_000) -> DatasetDict:
    """
    Expects the following folder layout:

        data_dir/
          metadata.csv          ← columns: file_name, sentence
          audio/
            utt001.wav
            utt002.wav
            ...

    Returns a HuggingFace DatasetDict with train / validation / test splits.
    """
    import pandas as pd

    meta_path = Path(data_dir) / "metadata.csv"
    if not meta_path.exists():
        print("⚠️  metadata.csv not found — creating a minimal demo dataset")
        return _create_demo_dataset(data_dir, sample_rate)

    df = pd.read_csv(meta_path)
    df["audio"] = df["file_name"].apply(
        lambda fn: str(Path(data_dir) / "audio" / fn)
    )

    dataset = Dataset.from_pandas(df[["audio", "sentence"]])
    dataset = dataset.cast_column("audio", Audio(sampling_rate=sample_rate))

    # Split
    split1 = dataset.train_test_split(test_size=0.15, seed=SEED)
    split2 = split1["test"].train_test_split(test_size=0.34, seed=SEED)

    return DatasetDict({
        "train"     : split1["train"],
        "validation": split2["train"],
        "test"      : split2["test"],
    })


def _create_demo_dataset(data_dir: str, sample_rate: int) -> DatasetDict:
    """Creates a tiny synthetic dataset for pipeline smoke-testing."""
    audio_dir = Path(data_dir) / "audio"
    audio_dir.mkdir(parents=True, exist_ok=True)

    samples = []
    for i in range(20):
        duration = np.random.randint(2, 8)
        audio = np.random.randn(duration * sample_rate).astype(np.float32) * 0.01
        path = audio_dir / f"demo_{i:04d}.wav"
        sf.write(str(path), audio, sample_rate)
        samples.append({"audio": str(path), "sentence": f"དཔེར་མཚོན། {i}"})

    dataset = Dataset.from_list(samples)
    dataset = dataset.cast_column("audio", Audio(sampling_rate=sample_rate))
    split = dataset.train_test_split(test_size=0.3, seed=SEED)
    return DatasetDict({"train": split["train"], "validation": split["test"], "test": split["test"]})


raw_datasets = load_local_audio_dataset(CFG["data_dir"], CFG["sample_rate"])
print(raw_datasets)
print(f"\n📊 Train samples     : {len(raw_datasets['train'])}")
print(f"📊 Validation samples: {len(raw_datasets['validation'])}")
print(f"📊 Test samples      : {len(raw_datasets['test'])}")

## 7. Load Processor & Model

In [ ]:
print(f"⏳ Loading processor and model: {CFG['model_name']} ...")

# ── Processor (feature extractor + tokenizer) ─────────────────────────────
processor = WhisperProcessor.from_pretrained(
    CFG["model_name"],
    language=CFG["language"],
    task=CFG["task"],
)

feature_extractor = processor.feature_extractor
tokenizer         = processor.tokenizer

# ── Model ─────────────────────────────────────────────────────────────────
model = WhisperForConditionalGeneration.from_pretrained(
    CFG["model_name"],
    torch_dtype=torch.bfloat16,      # bfloat16 for A100/Ada — saves ~50% VRAM vs fp32
    device_map="auto",               # Automatically place on GPU(s)
)

# Forced language/task tokens so the model never guesses
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=CFG["language"], task=CFG["task"]
)
model.config.suppress_tokens = []

# ── Gradient checkpointing ────────────────────────────────────────────────
if CFG["gradient_checkpointing"]:
    model.config.use_cache = False   # Incompatible with grad checkpointing
    model.gradient_checkpointing_enable()

# ── Parameter count ───────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
print(f"\n📦 Model loaded")
print(f"   Total parameters : {total_params/1e9:.2f}B")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"   VRAM allocated   : {allocated:.2f} GB")

## 8. Apply LoRA for Parameter-Efficient Fine-Tuning

In [ ]:
if CFG["use_lora"]:
    lora_config = LoraConfig(
        r             = CFG["lora_r"],
        lora_alpha    = CFG["lora_alpha"],
        lora_dropout  = CFG["lora_dropout"],
        bias          = "none",
        task_type     = TaskType.SEQ_2_SEQ_LM,
        # Target the attention projection layers in both encoder & decoder
        target_modules= ["q_proj", "v_proj", "k_proj", "out_proj",
                         "fc1", "fc2"],
    )
    model = get_peft_model(model, lora_config)

    trainable, total = 0, 0
    for p in model.parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()

    print(f"🔧 LoRA applied")
    print(f"   Trainable params : {trainable/1e6:.1f}M / {total/1e9:.2f}B  "
          f"({100*trainable/total:.2f}%)")
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        print(f"   VRAM allocated   : {allocated:.2f} GB")
else:
    print("ℹ️  Full fine-tuning (LoRA disabled)")

## 9. Feature Extraction & Preprocessing

In [ ]:
MAX_LABEL_LEN = 448   # Whisper decoder max length

def prepare_dataset(batch):
    """
    Convert raw audio → log-mel spectrogram  +  text → token IDs.
    Compatible with HuggingFace .map().
    """
    audio = batch["audio"]

    # ── Audio → features ─────────────────────────────────────────────
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="np",
    ).input_features[0]

    # ── Text → labels ────────────────────────────────────────────────
    batch["labels"] = tokenizer(
        basic_clean(batch["sentence"]),    # Apply basic cleaning
        max_length=MAX_LABEL_LEN,
        truncation=True,
    ).input_ids

    return batch


print("⚙️  Preprocessing datasets (this may take a few minutes)...")

processed_datasets = raw_datasets.map(
    prepare_dataset,
    remove_columns=raw_datasets["train"].column_names,
    num_proc=4,
    desc="Preprocessing",
)

print("✅ Preprocessing complete")
print(processed_datasets)

## 10. Data Collator

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Custom collator that:
    - Pads audio features to the same length
    - Pads label token IDs to the same length
    - Replaces padding token id with -100 so they are ignored in loss
    """
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Separate audio inputs and labels
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        # Pad audio features
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # Pad labels; replace padding with -100 (ignored in cross-entropy loss)
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Trim leading BOS token if present (Whisper adds it automatically)
        if (labels[:, 0] == self.decoder_start_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

print("✅ Data collator ready")

## 11. Evaluation Metric (Word Error Rate)

In [ ]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids     = pred.predictions
    label_ids    = pred.label_ids

    # Replace -100 padding back to pad token id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 2)}


print("✅ WER metric ready")

## 12. Training Arguments

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir                  = CFG["output_dir"],

    # ── Batch & Steps ────────────────────────────────────────────────
    per_device_train_batch_size = CFG["per_device_train_batch_size"],
    per_device_eval_batch_size  = CFG["per_device_eval_batch_size"],
    gradient_accumulation_steps = CFG["gradient_accumulation_steps"],
    max_steps                   = CFG["max_steps"],

    # ── Optimiser ────────────────────────────────────────────────────
    learning_rate               = CFG["learning_rate"],
    warmup_steps                = CFG["warmup_steps"],
    optim                       = "adamw_torch_fused",   # Fastest on CUDA
    lr_scheduler_type           = "cosine",

    # ── Precision ────────────────────────────────────────────────────
    fp16                        = CFG["fp16"],
    bf16                        = CFG["bf16"],
    gradient_checkpointing      = CFG["gradient_checkpointing"],

    # ── Logging / Saving ─────────────────────────────────────────────
    logging_steps               = CFG["logging_steps"],
    eval_strategy               = "steps",
    eval_steps                  = CFG["eval_steps"],
    save_steps                  = CFG["save_steps"],
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "wer",
    greater_is_better           = False,
    report_to                   = ["tensorboard"],

    # ── Generation (for evaluation) ──────────────────────────────────
    predict_with_generate       = True,
    generation_max_length       = MAX_LABEL_LEN,

    # ── DataLoader ───────────────────────────────────────────────────
    dataloader_num_workers      = CFG["dataloader_num_workers"],
    dataloader_pin_memory       = True,

    seed                        = SEED,
)

print("✅ Training arguments configured")
print(f"   Effective batch size: "
      f"{CFG['per_device_train_batch_size'] * CFG['gradient_accumulation_steps']}")

## 13. Trainer Setup & Training

In [ ]:
trainer = Seq2SeqTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = processed_datasets["train"],
    eval_dataset    = processed_datasets["validation"],
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    processing_class= processor.feature_extractor,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=5)],
)

print("🚀 Starting training...")
print(f"   Model  : {CFG['model_name']}")
print(f"   Lang   : {CFG['language']}  ({CFG['task']})")
print(f"   Steps  : {CFG['max_steps']}")
print("─" * 60)

train_result = trainer.train()

# ── Save metrics ─────────────────────────────────────────────────────────
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

print("\n✅ Training complete!")
print(train_result.metrics)

## 14. Evaluation on Test Set

In [ ]:
print("📊 Evaluating on test set...")
metrics = trainer.evaluate(
    eval_dataset    = processed_datasets["test"],
    metric_key_prefix="test",
)

trainer.log_metrics("test", metrics)
trainer.save_metrics("test", metrics)

print("\n📈 Test Results:")
for k, v in metrics.items():
    print(f"   {k:<30}: {v}")

## 15. Save Final Model

In [ ]:
save_path = Path(CFG["output_dir"]) / "final"

# Merge LoRA weights back into the base model (optional but recommended for deployment)
if CFG["use_lora"]:
    print("🔗 Merging LoRA adapters into base model...")
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(str(save_path))
else:
    trainer.save_model(str(save_path))

processor.save_pretrained(str(save_path))

print(f"💾 Model saved → {save_path}")

## 16. Inference — Transcribe New Audio / Extract from Documents

In [ ]:
from transformers import pipeline

# ── Load fine-tuned model for inference ──────────────────────────────────
asr_pipe = pipeline(
    task                 = "automatic-speech-recognition",
    model                = str(save_path),
    chunk_length_s       = 30,
    stride_length_s      = 5,
    device               = 0 if torch.cuda.is_available() else -1,
    torch_dtype          = torch.bfloat16,
    generate_kwargs      = {
        "language"   : CFG["language"],
        "task"       : CFG["task"],
        "num_beams"  : 5,
    }
)


def transcribe_audio(audio_path: str) -> str:
    """Transcribe a single audio file to Dzongkha text."""
    result = asr_pipe(audio_path)
    return result["text"]


def extract_language_from_doc_audio(doc_text: str, audio_path: Optional[str] = None) -> Dict:
    """
    Combined pipeline:
      - If audio provided: transcribe audio and align with document text
      - If only document: return cleaned OCR text
    Returns dict with transcript and matched_doc_text.
    """
    output = {"doc_text": basic_clean(doc_text)}

    if audio_path and Path(audio_path).exists():
        output["transcript"] = transcribe_audio(audio_path)

    return output


# ── Demo inference ───────────────────────────────────────────────────────
# Replace with a real .wav file path for actual inference:
demo_audio = list((Path(CFG["data_dir"]) / "audio").glob("*.wav"))

if demo_audio:
    sample_audio = str(demo_audio[0])
    print(f"🎤 Transcribing: {sample_audio}")
    transcript = transcribe_audio(sample_audio)
    print(f"📝 Transcript: {transcript}")
else:
    print("ℹ️  No audio files found for demo — add .wav files to data/bhutan/audio/")


# ── Demo document extraction ─────────────────────────────────────────────
if extracted_docs:
    sample_doc = extracted_docs[0]
    print(f"\n📄 Document: {sample_doc['filename']}")
    print(f"   Extracted text (first 200 chars): {sample_doc['text'][:200]}")

## 17. VRAM Profiling Utility

In [ ]:
def vram_report():
    """Print a summary of current VRAM usage."""
    if not torch.cuda.is_available():
        print("No CUDA GPU detected.")
        return

    for i in range(torch.cuda.device_count()):
        props     = torch.cuda.get_device_properties(i)
        total     = props.total_memory / 1e9
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved  = torch.cuda.memory_reserved(i) / 1e9
        free      = total - reserved

        print(f"GPU {i} — {props.name}")
        print(f"  Total    : {total:.2f} GB")
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved : {reserved:.2f} GB")
        print(f"  Free     : {free:.2f} GB")

vram_report()

---
## ✅ Training Summary

| Item | Value |
|------|-------|
| Base model | `openai/whisper-large-v3` (1.5B params) |
| Fine-tuning | LoRA (r=32, α=64) — ~20M trainable params |
| Language | Dzongkha (དྲི་མེད་གཞུང་ལུགས་ཀྱི་རྫོང་ཁ) |
| Precision | bfloat16 |
| VRAM footprint | ~28–30 GB of 32 GB |
| Document extraction | PyMuPDF + Tesseract OCR (dzo+eng) |
| Advanced cleaning | Commented out — activate in Cell 5 when needed |

> **Tip:** To activate advanced cleaning, uncomment the `advanced_clean` function in Cell 5
> and replace `basic_clean(...)` calls with `advanced_clean(...)` in `prepare_dataset`.